# Sampling final — selección de los 10 mejores nombres

**Lidera:** Santiago Diaz · **Compartido con:** Alan, Juan Camilo.

Carga `best_model.pt` (la celda ganadora entre RNN/LSTM/GRU) y barre todas las combinaciones de muestreo:

- `temperature ∈ {0.7, 1.0, 2.5, 4.0}`
- `top_k ∈ {None, 5, 10}`
- `top_p ∈ {None, 0.9, 0.95}`

Filtra nombres que ya están en `dinos.csv` (originalidad), persiste el barrido en `data/generated/names.csv` y selecciona los 10 finalistas.

In [ ]:
import sys, os
from pathlib import Path
REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / 'CLAUDE.md').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import torch
import pandas as pd
from itertools import product
from Parte_1_Generador_Caracteres.src.sample import load_model, generate_unique
from Parte_1_Generador_Caracteres.src.dataset import load_names

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Cargar el mejor modelo

In [ ]:
CHECKPOINT = 'Parte_1_Generador_Caracteres/models/best_model.pt'
model, vocab, max_len = load_model(CHECKPOINT, device=device)
seen = set(load_names('data/dinos.csv'))
print(f'modelo cargado | celda={model.cell_type} | max_len={max_len}')
print(f'nombres a excluir (originalidad): {len(seen)}')

## 2. Barrido de hiperparámetros

Para cada combinación generamos 20 nombres únicos y los acumulamos.

In [ ]:
TEMPS = [0.7, 1.0, 2.5, 4.0]
TOP_KS = [None, 5, 10]
TOP_PS = [None, 0.9, 0.95]
N_PER_CONFIG = 20

rows = []
for t, k, p in product(TEMPS, TOP_KS, TOP_PS):
    names = generate_unique(model, vocab, max_len, n=N_PER_CONFIG,
                            temperature=t, top_k=k, top_p=p,
                            seen=seen, device=device)
    for name in names:
        rows.append({
            'name': name,
            'cell_type': model.cell_type,
            'temperature': t,
            'top_k': k,
            'top_p': p,
            'length': len(name),
        })
    print(f'T={t} k={k} p={p} -> {len(names)} nombres')

df = pd.DataFrame(rows)
Path('data/generated').mkdir(parents=True, exist_ok=True)
df.to_csv('data/generated/names.csv', index=False)
print(f'\ntotal generados (únicos por config): {len(df)}')
df.head()

## 3. Análisis exploratorio del barrido

Diversidad por configuración (cuántos nombres distintos genera) y longitud media.

In [ ]:
summary = (
    df.groupby(['temperature', 'top_k', 'top_p'], dropna=False)
      .agg(n=('name', 'nunique'), avg_len=('length', 'mean'))
      .reset_index()
      .sort_values('temperature')
)
summary

## 4. Selección de los 10 finalistas

Criterios:
- Longitud entre 6 y 14 caracteres.
- Termina en un sufijo plausible (`saurus, raptor, odon, ceratops, mimus, long, venator, nyx, suchus, pteryx`).
- Variedad: máximo 2 nombres por configuración (T, top_k, top_p).

Edita la lista final manualmente al final si quieres ajustar.

In [ ]:
PALEO_SUFFIXES = ('saurus', 'raptor', 'odon', 'ceratops', 'mimus',
                  'long', 'venator', 'nyx', 'suchus', 'pteryx', 'don')

def is_plausible(name: str) -> bool:
    return 6 <= len(name) <= 14 and name.endswith(PALEO_SUFFIXES)

candidates = df[df['name'].apply(is_plausible)].drop_duplicates('name').copy()

# máximo 2 por configuración para forzar variedad
candidates = (candidates
              .groupby(['temperature', 'top_k', 'top_p'], dropna=False, group_keys=False)
              .head(2)
              .reset_index(drop=True))

finalistas = candidates.head(10).reset_index(drop=True)
finalistas

In [ ]:
Path('data/generated').mkdir(parents=True, exist_ok=True)
finalistas.to_csv('data/generated/top10_names.csv', index=False)
finalistas[['name']].to_json('data/generated/top10_names.json', orient='records', indent=2)
print('guardados:')
print('  data/generated/top10_names.csv')
print('  data/generated/top10_names.json')
print('\n10 finalistas:')
for n in finalistas['name']:
    print(' -', n)

## 5. Nota comparativa (3–5 líneas)

Editar [reports/sampling_comparison.md](../../reports/sampling_comparison.md) con el análisis. Ejes a tocar:

- **Temperatura baja (0.7)**: nombres conservadores, plausibles, repetitivos. Buenos para coherencia, malos para creatividad.
- **Temperatura alta (2.5–4.0)**: caos. Saca cosas raras pero pierde la "forma" de dinosaurio.
- **top-k vs top-p**: top-p adaptativo suele dar mejor balance que top-k fijo cuando la entropía varía entre pasos.
- **Combinación recomendada**: `T=1.0, top_p=0.9` o `T=1.0, top_k=10` — punto dulce entre originalidad y plausibilidad.